In [ ]:
#Heatmap_visualisation_upregulated_GOs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import textwrap

# ============================================================
# Settings
# ============================================================

base_dir = Path(".")

gprofiler_files = {
    "SA vs Mock": "./GOprofiler/SA_Mock_up1/gProfiler_09022026_intersections_SA_M.csv",
    "NSC+SA vs Mock": "./GOprofiler/NSCSA_Mock_up1/gProfiler_09022026_intersections_NSCSA_M.csv",
    "V1/C3dsR+SA vs Mock": "./GOprofiler/V1SA_Mock_up1/gProfiler_09022026_intersections_V1SA_M.csv",
    "V1/C3dsR vs Mock": "./GOprofiler/V1_Mock_up1/gProfiler_09022026_intersections_V1_M.csv"
}

padj_cut = 0.05

# Keep only g:Profiler-highlighted terms
USE_HIGHLIGHTED_ONLY = True

# Limit number of terms for a clean paper figure
MAX_TERMS = 20

# Optional: keep only selected source types
sources_to_keep = ["GO:BP", "GO:MF", "GO:CC", "KEGG", "REAC", "WP"]

# ============================================================
# Read g:Profiler files
# ============================================================

all_rows = []

for comparison, filename in gprofiler_files.items():
    file_path = base_dir / filename

    if not file_path.exists():
        raise FileNotFoundError(f"Cannot find: {file_path}")

    df = pd.read_csv(file_path)
    df["comparison"] = comparison

    required_cols = {
        "source",
        "term_name",
        "term_id",
        "highlighted",
        "adjusted_p_value",
        "intersection_size"
    }

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{filename} is missing columns: {missing}")

    # Robust highlighted conversion
    if df["highlighted"].dtype == bool:
        df["highlighted_bool"] = df["highlighted"]
    else:
        df["highlighted_bool"] = (
            df["highlighted"]
            .astype(str)
            .str.lower()
            .isin(["true", "t", "1", "yes"])
        )

    # Calculate -log10 adjusted p-value
    if "negative_log10_of_adjusted_p_value" in df.columns:
        df["neglog10_padj"] = df["negative_log10_of_adjusted_p_value"]
    else:
        df["neglog10_padj"] = -np.log10(df["adjusted_p_value"])

    all_rows.append(df)

go = pd.concat(all_rows, ignore_index=True)

# ============================================================
# Filter terms
# ============================================================

# Significant terms only
go = go[go["adjusted_p_value"] < padj_cut].copy()

# Keep only selected sources
go = go[go["source"].isin(sources_to_keep)].copy()

# Keep only g:Profiler highlighted terms
if USE_HIGHLIGHTED_ONLY:
    go = go[go["highlighted_bool"] == True].copy()

# Label for plotting
go["term_label"] = go["source"] + ": " + go["term_name"]

print("Number of significant highlighted terms per comparison:")
print(go.groupby("comparison").size())

# ============================================================
# Rank and select terms
# Keep top highlighted terms from each comparison
# ============================================================

if go.empty:
    raise ValueError(
        "No GO terms remain after filtering. "
        "Try setting USE_HIGHLIGHTED_ONLY = False or increasing padj_cut."
    )

TOP_PER_COMPARISON = 8
MAX_TERMS = 25

selected_term_ids = []

for comparison in gprofiler_files.keys():
    sub = go[go["comparison"] == comparison].copy()

    if sub.empty:
        print(f"No significant highlighted terms for {comparison}")
        continue

    sub = sub.sort_values(
        ["adjusted_p_value", "intersection_size"],
        ascending=[True, False]
    )

    top_terms = sub.head(TOP_PER_COMPARISON)["term_id"].tolist()
    selected_term_ids.extend(top_terms)

# remove duplicates while keeping order
selected_term_ids = list(dict.fromkeys(selected_term_ids))

# rank selected terms overall
term_rank = (
    go[go["term_id"].isin(selected_term_ids)]
    .groupby(["term_id", "term_label"], as_index=False)
    .agg(
        best_padj=("adjusted_p_value", "min"),
        max_neglog10_padj=("neglog10_padj", "max"),
        total_intersection=("intersection_size", "sum"),
        present_in_n=("comparison", "nunique")
    )
)

term_rank = term_rank.sort_values(
    ["present_in_n", "max_neglog10_padj", "total_intersection"],
    ascending=[False, False, False]
)

if MAX_TERMS is not None:
    term_rank = term_rank.head(MAX_TERMS)

selected_order = term_rank["term_id"].tolist()

go_sel = go[go["term_id"].isin(selected_order)].copy()
# ============================================================
# Build heatmap matrices
# ============================================================

comparison_order = list(gprofiler_files.keys())

score_mat = (
    go_sel.pivot_table(
        index="term_id",
        columns="comparison",
        values="neglog10_padj",
        aggfunc="max"
    )
)

count_mat = (
    go_sel.pivot_table(
        index="term_id",
        columns="comparison",
        values="intersection_size",
        aggfunc="max"
    )
)

# Safer: keep all comparison columns, even if blank
score_mat = score_mat.reindex(index=selected_order, columns=comparison_order)
count_mat = count_mat.reindex(index=selected_order, columns=comparison_order)

score_mat = score_mat[comparison_order]
count_mat = count_mat[comparison_order]

# Term labels
label_map = (
    go_sel.drop_duplicates("term_id")
          .set_index("term_id")["term_label"]
          .to_dict()
)

row_labels = [label_map.get(term_id, term_id) for term_id in score_mat.index]

def wrap_label(x, width=48):
    return "\n".join(textwrap.wrap(str(x), width=width))

score_mat.index = [wrap_label(x, width=48) for x in row_labels]
count_mat.index = score_mat.index

# Annotation text
annot_mat = count_mat.copy()
annot_mat = annot_mat.map(lambda x: "" if pd.isna(x) else str(int(x)))

# Hide non-significant / absent cells
mask = score_mat.isna()
plot_mat = score_mat.fillna(0)


# Save matrices
score_mat.to_csv("GO_highlighted_heatmap_neglog10_padj_matrix_up.csv")
count_mat.to_csv("GO_highlighted_heatmap_intersection_size_matrix_up.csv")
term_rank.to_csv("GO_highlighted_terms_used_for_heatmap_up.csv", index=False)

# ============================================================
# Plot heatmap
# ============================================================

plt.figure(figsize=(8.5, max(5, 0.48 * len(score_mat))))

ax = sns.heatmap(
    plot_mat,
    mask=mask,
    cmap="YlOrRd",
    linewidths=0.6,
    linecolor="lightgrey",
    annot=annot_mat,
    fmt="",
    cbar_kws={"label": "-log10(adjusted p-value)"},
    vmin=0
)

ax.set_title(
    "Comparative GO and pathway enrichment analysis of up-regulated DEGs",
    fontsize=12,
    fontweight="bold",
    pad=12
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.xticks(rotation=35, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=8)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color("black")

plt.tight_layout()

plt.savefig("GO_highlighted_comparative_heatmap_up.png", dpi=600, bbox_inches="tight")
plt.savefig("GO_highlighted_comparative_heatmap_up.pdf", bbox_inches="tight")

plt.show()

In [ ]:
#Heatmap_downregulated_GOs
# ============================================================
# Settings
# ============================================================

base_dir = Path(".")

gprofiler_files = {
    "SA vs Mock": "./GOprofiler/SA_Mock_dw1/gProfiler_09022026_intersections_SA_M_dw.csv",
    "NSC+SA vs Mock": "./GOprofiler/NSCSA_Mock_dw1/gProfiler_09022026_intersections_NSCSA_M_dw.csv",
    "V1/C3dsR+SA vs Mock": "./GOprofiler/V1SA_Mock_dw1/gProfiler_09022026_intersections_V1SA_M_dw.csv",
    "NSC vs Mock": "./GOprofiler/NSC_Mock_dw1/gProfiler_09022026_intersections_NSC_M_dw.csv",
    "V1 vs Mock": "./GOprofiler/V1_Mock_dw1/gProfiler_09022026_intersections_V1_M_dw.csv"
}

padj_cut = 0.05

# Keep only g:Profiler-highlighted terms
USE_HIGHLIGHTED_ONLY = True

# Limit number of terms for a clean paper figure
MAX_TERMS = 20

# Optional: keep only selected source types
sources_to_keep = ["GO:BP", "GO:MF", "GO:CC", "KEGG", "REAC", "WP"]

# ============================================================
# Read g:Profiler files
# ============================================================

all_rows = []

for comparison, filename in gprofiler_files.items():
    file_path = base_dir / filename

    if not file_path.exists():
        raise FileNotFoundError(f"Cannot find: {file_path}")

    df = pd.read_csv(file_path)
    df["comparison"] = comparison

    required_cols = {
        "source",
        "term_name",
        "term_id",
        "highlighted",
        "adjusted_p_value",
        "intersection_size"
    }

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{filename} is missing columns: {missing}")

    # Robust highlighted conversion
    if df["highlighted"].dtype == bool:
        df["highlighted_bool"] = df["highlighted"]
    else:
        df["highlighted_bool"] = (
            df["highlighted"]
            .astype(str)
            .str.lower()
            .isin(["true", "t", "1", "yes"])
        )

    # Calculate -log10 adjusted p-value
    if "negative_log10_of_adjusted_p_value" in df.columns:
        df["neglog10_padj"] = df["negative_log10_of_adjusted_p_value"]
    else:
        df["neglog10_padj"] = -np.log10(df["adjusted_p_value"])

    all_rows.append(df)

go = pd.concat(all_rows, ignore_index=True)

# ============================================================
# Filter terms
# ============================================================

# Significant terms only
go = go[go["adjusted_p_value"] < padj_cut].copy()

# Keep only selected sources
go = go[go["source"].isin(sources_to_keep)].copy()

# Keep only g:Profiler highlighted terms
if USE_HIGHLIGHTED_ONLY:
    go = go[go["highlighted_bool"] == True].copy()

# Label for plotting
go["term_label"] = go["source"] + ": " + go["term_name"]

print("Number of significant highlighted terms per comparison:")
print(go.groupby("comparison").size())

# ============================================================
# Rank and select terms
# Keep top highlighted terms from each comparison
# ============================================================

if go.empty:
    raise ValueError(
        "No GO terms remain after filtering. "
        "Try setting USE_HIGHLIGHTED_ONLY = False or increasing padj_cut."
    )

TOP_PER_COMPARISON = 8
MAX_TERMS = 25

selected_term_ids = []

for comparison in gprofiler_files.keys():
    sub = go[go["comparison"] == comparison].copy()

    if sub.empty:
        print(f"No significant highlighted terms for {comparison}")
        continue

    sub = sub.sort_values(
        ["adjusted_p_value", "intersection_size"],
        ascending=[True, False]
    )

    top_terms = sub.head(TOP_PER_COMPARISON)["term_id"].tolist()
    selected_term_ids.extend(top_terms)

# remove duplicates while keeping order
selected_term_ids = list(dict.fromkeys(selected_term_ids))

# rank selected terms overall
term_rank = (
    go[go["term_id"].isin(selected_term_ids)]
    .groupby(["term_id", "term_label"], as_index=False)
    .agg(
        best_padj=("adjusted_p_value", "min"),
        max_neglog10_padj=("neglog10_padj", "max"),
        total_intersection=("intersection_size", "sum"),
        present_in_n=("comparison", "nunique")
    )
)

term_rank = term_rank.sort_values(
    ["present_in_n", "max_neglog10_padj", "total_intersection"],
    ascending=[False, False, False]
)

if MAX_TERMS is not None:
    term_rank = term_rank.head(MAX_TERMS)

selected_order = term_rank["term_id"].tolist()

go_sel = go[go["term_id"].isin(selected_order)].copy()
# ============================================================
# Build heatmap matrices
# ============================================================

comparison_order = list(gprofiler_files.keys())

score_mat = (
    go_sel.pivot_table(
        index="term_id",
        columns="comparison",
        values="neglog10_padj",
        aggfunc="max"
    )
)

count_mat = (
    go_sel.pivot_table(
        index="term_id",
        columns="comparison",
        values="intersection_size",
        aggfunc="max"
    )
)

# Safer: keep all comparison columns, even if blank
score_mat = score_mat.reindex(index=selected_order, columns=comparison_order)
count_mat = count_mat.reindex(index=selected_order, columns=comparison_order)

score_mat = score_mat[comparison_order]
count_mat = count_mat[comparison_order]

# Term labels
label_map = (
    go_sel.drop_duplicates("term_id")
          .set_index("term_id")["term_label"]
          .to_dict()
)

row_labels = [label_map.get(term_id, term_id) for term_id in score_mat.index]

def wrap_label(x, width=48):
    return "\n".join(textwrap.wrap(str(x), width=width))

score_mat.index = [wrap_label(x, width=48) for x in row_labels]
count_mat.index = score_mat.index

# Annotation text
annot_mat = count_mat.copy()
annot_mat = annot_mat.map(lambda x: "" if pd.isna(x) else str(int(x)))

# Hide non-significant / absent cells
mask = score_mat.isna()
plot_mat = score_mat.fillna(0)

# Save matrices
score_mat.to_csv("GO_highlighted_heatmap_neglog10_padj_matrix_dw.csv")
count_mat.to_csv("GO_highlighted_heatmap_intersection_size_matrix_dw.csv")
term_rank.to_csv("GO_highlighted_terms_used_for_heatmap_dw.csv", index=False)

# ============================================================
# Plot heatmap
# ============================================================

plt.figure(figsize=(8.5, max(5, 0.48 * len(score_mat))))

ax = sns.heatmap(
    plot_mat,
    mask=mask,
    cmap="YlOrRd",
    linewidths=0.6,
    linecolor="lightgrey",
    annot=annot_mat,
    fmt="",
    cbar_kws={"label": "-log10(adjusted p-value)"},
    vmin=0
)

ax.set_title(
    "Comparative GO and pathway enrichment analysis of down-regulated DEGs",
    fontsize=12,
    fontweight="bold",
    pad=12
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.xticks(rotation=35, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=8)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color("black")

plt.tight_layout()

plt.savefig("GO_highlighted_comparative_heatmap_dw.png", dpi=600, bbox_inches="tight")
plt.savefig("GO_highlighted_comparative_heatmap_dw.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# Extract genes from highlighted g:Profiler terms for further merge annotation
import pandas as pd
import re
from pathlib import Path

# ============================================================
# Your file locations
# ============================================================

gprofiler_files = {
    "SA_vs_Mock": "./GOprofiler/SA_Mock_up1/gProfiler_09022026_intersections_SA_M.csv",
    "NSC+SA_vs_Mock": "./GOprofiler/NSCSA_Mock_up1/gProfiler_09022026_intersections_NSCSA_M.csv",
    "V1/C3dsR+SA_vs_Mock": "./GOprofiler/V1SA_Mock_up1/gProfiler_09022026_intersections_V1SA_M.csv"
}

de_files = {
    "SA_vs_Mock": "./python_sorted_result_fromDeseq2/SA_vs_Mock_DE_annotated.csv",
    "NSC+SA_vs_Mock": "./python_sorted_result_fromDeseq2/NSCSA_vs_Mock_DE_annotated.csv",
    "V1/C3dsR+SA_vs_Mock": "./python_sorted_result_fromDeseq2/SpecSA_vs_Mock_DE_annotated.csv"
}

# ============================================================
# Helper functions
# ============================================================

def clean_solyc_id(x):
    """
    Convert IDs like:
    SOLYC04G007070
    Solyc04g007070
    gene:Solyc04g007070.3.1
    to:
    Solyc04g007070
    """
    if pd.isna(x):
        return None

    x = str(x)
    m = re.search(r"(SOLYC\d{2}G\d{6}|Solyc\d{2}g\d{6})", x)

    if not m:
        return None

    root = m.group(1).upper()  # SOLYC04G007070
    root = root.replace("SOLYC", "Solyc").replace("G", "g")
    return root


def split_gene_list(x):
    """
    Split g:Profiler intersection gene list.
    Handles comma, semicolon, or mixed separators.
    """
    if pd.isna(x) or str(x).strip() == "":
        return []

    genes = re.split(r"[,;]\s*", str(x).strip())
    genes = [g.strip() for g in genes if g.strip()]
    return genes


def get_intersection_column(df):
    """
    Find the g:Profiler gene intersection column.
    """
    possible_cols = ["intersections", "intersection", "query_intersections"]

    for col in possible_cols:
        if col in df.columns:
            return col

    raise ValueError(
        f"Cannot find intersection column. Available columns are:\n{df.columns.tolist()}"
    )


def highlighted_to_bool(series):
    """
    Convert g:Profiler highlighted column to boolean.
    """
    return (
        series.astype(str)
        .str.lower()
        .isin(["true", "t", "1", "yes"])
    )

# ============================================================
# 1. Extract genes from highlighted g:Profiler terms only
# ============================================================

rows = []

for comparison, file_path in gprofiler_files.items():
    df = pd.read_csv(file_path)

    print("\n==============================")
    print("Comparison:", comparison)
    print("Columns:", df.columns.tolist())

    intersection_col = get_intersection_column(df)

    df["highlighted_bool"] = highlighted_to_bool(df["highlighted"])

    df_h = df[df["highlighted_bool"] == True].copy()

    print("Number of highlighted terms:", len(df_h))

    if len(df_h) > 0:
        print(df_h[["source", "term_name", "adjusted_p_value", "intersection_size"]])

    for _, row in df_h.iterrows():
        genes = split_gene_list(row[intersection_col])

        for gene in genes:
            rows.append({
                "comparison": comparison,
                "source": row["source"],
                "term_id": row["term_id"],
                "term_name": row["term_name"],
                "highlighted": True,
                "adjusted_p_value_GO": row["adjusted_p_value"],
                "neglog10_GO_padj": row.get(
                    "negative_log10_of_adjusted_p_value",
                    None
                ),
                "intersection_size": row["intersection_size"],
                "gene_from_gProfiler": gene,
                "Solyc_root": clean_solyc_id(gene)
            })

highlighted_genes = pd.DataFrame(rows)

print("\n==============================")
print("Total highlighted GO-gene rows:", len(highlighted_genes))

if len(highlighted_genes) > 0:
    print("Comparisons in highlighted_genes:")
    print(highlighted_genes["comparison"].value_counts())

highlighted_genes.to_csv(
    "Highlighted_gProfiler_GO_gene_membership_upregulated.csv",
    index=False
)

In [ ]:
# ============================================================
# 2. Merge highlighted GO genes with DESeq2 annotation
# ============================================================

de_rows = []

for comparison, file_path in de_files.items():
    de = pd.read_csv(file_path).copy()

    if "GeneID" in de.columns:
        de["Solyc_root"] = de["GeneID"].apply(clean_solyc_id)
    elif "gene" in de.columns:
        de["Solyc_root"] = de["gene"].apply(clean_solyc_id)
    else:
        raise ValueError(f"No GeneID or gene column found in {file_path}")

    de["comparison"] = comparison

    keep_cols = [
        "comparison",
        "Solyc_root",
        "gene",
        "GeneID",
        "Description",
        "baseMean",
        "log2FoldChange",
        "padj",
        "pvalue"
    ]

    keep_cols = [c for c in keep_cols if c in de.columns]
    de_rows.append(de[keep_cols])

de_all = pd.concat(de_rows, ignore_index=True)

highlighted_annotated = highlighted_genes.merge(
    de_all,
    on=["comparison", "Solyc_root"],
    how="left"
)

highlighted_annotated.to_csv(
    "Highlighted_GO_genes_with_DESeq2_annotation_upregulated.csv",
    index=False
)

print("Merged rows:", len(highlighted_annotated))
print("Rows with DESeq2 annotation:", highlighted_annotated["log2FoldChange"].notna().sum())

highlighted_annotated.head()

In [ ]:
# ============================================================
# 3. Find V1/C3dsR+SA-unique genes among highlighted GO terms
# ============================================================

sa_genes = set(
    highlighted_annotated.loc[
        highlighted_annotated["comparison"] == "SA_vs_Mock",
        "Solyc_root"
    ].dropna()
)

nscsa_genes = set(
    highlighted_annotated.loc[
        highlighted_annotated["comparison"] == "NSC+SA_vs_Mock",
        "Solyc_root"
    ].dropna()
)

v1sa_genes = set(
    highlighted_annotated.loc[
        highlighted_annotated["comparison"] == "V1/C3dsR+SA_vs_Mock",
        "Solyc_root"
    ].dropna()
)

v1_unique = v1sa_genes - sa_genes - nscsa_genes

print("SA highlighted genes:", len(sa_genes))
print("NSC+SA highlighted genes:", len(nscsa_genes))
print("V1/C3dsR+SA highlighted genes:", len(v1sa_genes))
print("V1/C3dsR+SA unique highlighted genes:", len(v1_unique))
print(v1_unique)

v1_unique_table = highlighted_annotated[
    (highlighted_annotated["comparison"] == "V1/C3dsR+SA_vs_Mock") &
    (highlighted_annotated["Solyc_root"].isin(v1_unique))
].copy()

v1_unique_table = v1_unique_table.sort_values(
    ["term_name", "log2FoldChange"],
    ascending=[True, False]
)

v1_unique_table.to_csv(
    "V1C3dsR_SA_unique_highlighted_GO_genes_annotated_upregulated.csv",
    index=False
)

v1_unique_table[
    [
        "comparison",
        "source",
        "term_name",
        "Solyc_root",
        "Description",
        "log2FoldChange",
        "padj"
    ]
]

In [ ]:

# ============================================================
# Input files
# ============================================================

de_files = {
    "SA vs Mock": "./python_sorted_result_fromDeseq2/SA_vs_Mock_DE_annotated.csv",
    "NSC dsRNA+SA vs Mock": "./python_sorted_result_fromDeseq2/NSCSA_vs_Mock_DE_annotated.csv",
    "V1/C3dsR+SA vs Mock": "./python_sorted_result_fromDeseq2/SpecSA_vs_Mock_DE_annotated.csv"
}

# Defence-associated genes of interest
target_genes = [
    "Solyc08g023660.3",
    "Solyc05g007630.3",
    "Solyc05g046140.3",
    "Solyc04g007010.3",
    "Solyc12g096960.2",
    "Solyc12g096920.1",
    "Solyc07g006770.2",
    "Solyc09g092310.1",
    "Solyc04g007820.3",
    "Solyc04g007070.3",
    "Solyc10g084860.1",
    "Solyc07g049700.1",
    "Solyc12g005620.1"
]

# ============================================================
# Helper functions
# ============================================================

def clean_solyc_id(x):
    """
    Convert:
    Solyc04g007070.3
    gene:Solyc04g007070.3.1
    SOLYC04G007070
    to:
    Solyc04g007070
    """
    if pd.isna(x):
        return None

    x = str(x)
    match = re.search(r"(Solyc\d{2}g\d{6}|SOLYC\d{2}G\d{6})", x)

    if not match:
        return None

    root = match.group(1).upper()
    root = root.replace("SOLYC", "Solyc").replace("G", "g")
    return root


def p_to_star(p):
    """
    Significance stars based on adjusted p-value.
    """
    if pd.isna(p):
        return ""
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""


def clean_description(desc, max_len=28):
    """
    Remove text in parentheses and shorten annotation.
    """
    if pd.isna(desc):
        return ""

    desc = str(desc)

    # Remove anything inside parentheses
    desc = re.sub(r"\s*\(.*?\)", "", desc)

    # Remove common words that make labels too long
    remove_words = [
        "putative", "Putative",
        "protein", "Protein",
        "family", "Family"
    ]

    for word in remove_words:
        desc = desc.replace(word, "")

    # Clean spacing
    desc = re.sub(r"\s+", " ", desc).strip()

    # Shorten
    if len(desc) > max_len:
        desc = desc[:max_len].rstrip() + "..."

    return desc


# ============================================================
# Clean target gene IDs
# ============================================================

target_roots = [clean_solyc_id(g) for g in target_genes]

# ============================================================
# Read DESeq2 files and extract target genes
# ============================================================

records = []

for comparison, file_path in de_files.items():
    df = pd.read_csv(file_path)

    if "GeneID" in df.columns:
        df["Solyc_root"] = df["GeneID"].apply(clean_solyc_id)
    elif "gene" in df.columns:
        df["Solyc_root"] = df["gene"].apply(clean_solyc_id)
    else:
        raise ValueError(f"No GeneID or gene column found in {file_path}")

    sub = df[df["Solyc_root"].isin(target_roots)].copy()
    sub["comparison"] = comparison

    records.append(sub)

all_de = pd.concat(records, ignore_index=True)

if all_de.empty:
    raise ValueError("No target genes were found in the DESeq2 files. Check gene IDs and file paths.")

# Save extracted DESeq2 table
all_de.to_csv("Selected_defence_genes_DESeq2_results.csv", index=False)

print("Extracted rows:", len(all_de))
print("Extracted genes:", all_de["Solyc_root"].nunique())

missing_genes = sorted(set(target_roots) - set(all_de["Solyc_root"].dropna()))
if missing_genes:
    print("Missing target genes:")
    print(missing_genes)

# ============================================================
# Build heatmap matrices
# ============================================================

comparison_order = list(de_files.keys())

# log2FC matrix
log2fc_mat = (
    all_de
    .pivot_table(
        index="Solyc_root",
        columns="comparison",
        values="log2FoldChange",
        aggfunc="first"
    )
    .reindex(target_roots)
    .reindex(columns=comparison_order)
)

# padj matrix
padj_mat = (
    all_de
    .pivot_table(
        index="Solyc_root",
        columns="comparison",
        values="padj",
        aggfunc="first"
    )
    .reindex(target_roots)
    .reindex(columns=comparison_order)
)

# Description map
if "Description" in all_de.columns:
    desc_map = (
        all_de
        .drop_duplicates("Solyc_root")
        .set_index("Solyc_root")["Description"]
        .to_dict()
    )
else:
    desc_map = {}

# ============================================================
# Sort genes by V1/C3dsR+SA response
# ============================================================

order_col = "V1/C3dsR+SA vs Mock"

gene_order = (
    log2fc_mat[order_col]
    .sort_values(ascending=False)
    .index
)

log2fc_mat = log2fc_mat.loc[gene_order]
padj_mat = padj_mat.loc[gene_order]

# Significance matrix
star_mat = padj_mat.map(p_to_star)

# Short row labels
row_labels = []

for gene in log2fc_mat.index:
    desc = desc_map.get(gene, "")
    short_desc = clean_description(desc, max_len=28)

    if short_desc:
        row_labels.append(f"{gene}\n{short_desc}")
    else:
        row_labels.append(gene)

# ============================================================
# Plot matrix
# ============================================================

plot_mat = log2fc_mat.copy()
mask_missing = plot_mat.isna()

# Symmetric colour limit
vmax = np.nanmax(np.abs(plot_mat.to_numpy()))
vmax = max(2, np.ceil(vmax))

plot_mat = plot_mat.fillna(0)

# ============================================================
# Plot heatmap
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.2, 0.55 * len(log2fc_mat) + 1.8)
)

im = ax.imshow(
    plot_mat,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax
)

# X-axis labels
ax.set_xticks(np.arange(len(comparison_order)))
ax.set_xticklabels(
    [
        "SA",
        "NSC+SA",
        "V1/C3dsR+SA"
    ],
    rotation=30,
    ha="right",
    fontsize=10
)

# Y-axis labels
ax.set_yticks(np.arange(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=10)

# Add log2FC values plus significance stars
for i in range(plot_mat.shape[0]):
    for j in range(plot_mat.shape[1]):

        if mask_missing.iloc[i, j]:
            text = ""
            text_color = "black"
            fontweight = "normal"
        else:
            fc = log2fc_mat.iloc[i, j]
            star = star_mat.iloc[i, j]
            text = f"{fc:.1f}{star}"

            # Use white text on very strong colours for readability
            text_color = "white" if abs(fc) > 0.65 * vmax else "black"
            fontweight = "bold" if star != "" else "normal"

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=10,
            fontweight=fontweight,
            color=text_color
        )

# Grid lines
ax.set_xticks(np.arange(-0.5, len(comparison_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_labels), 1), minor=True)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=1.2
)

ax.tick_params(which="minor", bottom=False, left=False)

# Remove heavy frame
for spine in ax.spines.values():
    spine.set_visible(False)

# Colour bar
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label("log2 fold change", fontsize=12)
cbar.ax.tick_params(labelsize=9)

# Title
ax.set_title(
    "Defence-associated genes",
    fontsize=14,
    fontweight="bold",
    pad=14
)

plt.tight_layout()

# Save figure
plt.savefig(
    "Selected_defence_genes_log2FC_heatmap_pretty.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "Selected_defence_genes_log2FC_heatmap_pretty.pdf",
    bbox_inches="tight"
)

plt.show()